In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from Functions import (laydowncoordinates, compute_A, energy, propose_move, valid_fold)

np.random.seed(0)  # optional, for reproducibility across reruns

H = 1
P = 0
Gamma = [H, P, H, P, P, H, H, P, H, P, P, H, P, H, H, P, P, H, P, H]
A = compute_A(Gamma)
N = len(Gamma)

In [2]:
def run_sa_trial(Gamma, A, N, beta, propose_move, laydowncoordinates, valid_fold, energy, H):
    while True:
        d = np.random.randint(0, 4, size=N - 1)
        coordinates = laydowncoordinates(d)
        if valid_fold(coordinates):
            break

    E = energy(coordinates, Gamma, A, H)
    best_E = E

    for i in range(len(beta)):
        while True:
            d_new = propose_move(d)
            coords_new = laydowncoordinates(d_new)
            if valid_fold(coords_new):
                break
        E_new = energy(coords_new, Gamma, A, H)
        deltaE = E_new - E

        p_accept = 1.0 / (1.0 + np.exp(deltaE * beta[i]))
        if np.random.rand() < p_accept:
            d, coordinates, E = d_new, coords_new, E
            E = E_new
            if E < best_E:
                best_E = E

    return best_E, E

In [3]:
def schedules(num_sweeps, reinforce_betas=None):
    return {
        "Linear": np.linspace(0.1, 10, num_sweeps),
        "Geometric": 0.1 * (10 / 0.1) ** (np.arange(num_sweeps) / (num_sweeps - 1)),
        "Logarithmic": 0.1 + (10 - 0.1) * np.log1p(np.arange(num_sweeps)) / np.log1p(num_sweeps - 1),
        "Linear Staircase": np.repeat(np.linspace(0.1, 10, 20), num_sweeps // 20),
        "Geometric Staircase": np.repeat(0.1 * (10 / 0.1) ** (np.arange(20) / 19), num_sweeps // 20),
        "Quenching": np.full(num_sweeps, 10.0),
        "REINFORCE": reinforce_betas,  # None for now — will skip automatically
    }

In [4]:
test_beta = np.linspace(0.1, 10, 100)
t0 = time.time()
best_E, final_E = run_sa_trial(Gamma, A, N, test_beta, propose_move,
                                laydowncoordinates, valid_fold, energy, H)
elapsed = time.time() - t0
print(f"one trial (100 sweeps) took {elapsed:.4f}s -> best_E={best_E}, final_E={final_E}")

num_trials = 1000
num_sweeps = 10000
num_schedules = 6  # excluding REINFORCE for now
est_per_trial = elapsed * (num_sweeps / 100)
est_total = est_per_trial * num_trials * num_schedules
print(f"estimated total time for full benchmark: {est_total/60:.1f} minutes")

one trial (100 sweeps) took 0.0407s -> best_E=-4.0, final_E=-4.0
estimated total time for full benchmark: 407.4 minutes


In [5]:
target_E = -9
num_trials = 1000
num_sweeps = 10000
PRINT_EVERY = 100  # print progress every N trials (adjust as you like)

sched_dict = schedules(num_sweeps, reinforce_betas=None)  # REINFORCE not ready yet
results = {name: {"best": [], "final": []} for name in sched_dict.keys()}

overall_start = time.time()

for name, beta in sched_dict.items():
    if beta is None:
        print(f"skipping {name} (no schedule provided)")
        continue

    print(f"\n=== running {name} ({num_trials} trials, {num_sweeps} sweeps each) ===")
    schedule_start = time.time()

    for trial in range(num_trials):
        best_E, final_E = run_sa_trial(Gamma, A, N, beta, propose_move,
                                        laydowncoordinates, valid_fold, energy, H)
        results[name]["best"].append(best_E)
        results[name]["final"].append(final_E)

        if (trial + 1) % PRINT_EVERY == 0 or (trial + 1) == num_trials:
            elapsed = time.time() - schedule_start
            rate = (trial + 1) / elapsed
            remaining = (num_trials - (trial + 1)) / rate
            running_best_mean = np.mean(results[name]["best"])
            print(f"  [{name}] trial {trial+1}/{num_trials} "
                  f"({elapsed:.1f}s elapsed, ~{remaining:.1f}s remaining) "
                  f"running mean best_E={running_best_mean:.3f}")

    print(f"=== {name} done in {time.time()-schedule_start:.1f}s ===")

print(f"\nTOTAL benchmark time: {(time.time()-overall_start)/60:.1f} minutes")


=== running Linear (1000 trials, 10000 sweeps each) ===
  [Linear] trial 100/1000 (336.1s elapsed, ~3025.2s remaining) running mean best_E=-7.140
  [Linear] trial 200/1000 (687.6s elapsed, ~2750.4s remaining) running mean best_E=-7.210
  [Linear] trial 300/1000 (1061.3s elapsed, ~2476.4s remaining) running mean best_E=-7.170
  [Linear] trial 400/1000 (1199.0s elapsed, ~1798.5s remaining) running mean best_E=-7.213
  [Linear] trial 500/1000 (1331.6s elapsed, ~1331.6s remaining) running mean best_E=-7.218
  [Linear] trial 600/1000 (1525.9s elapsed, ~1017.3s remaining) running mean best_E=-7.208
  [Linear] trial 700/1000 (1714.1s elapsed, ~734.6s remaining) running mean best_E=-7.201
  [Linear] trial 800/1000 (1907.5s elapsed, ~476.9s remaining) running mean best_E=-7.209
  [Linear] trial 900/1000 (2087.7s elapsed, ~232.0s remaining) running mean best_E=-7.193
  [Linear] trial 1000/1000 (2254.1s elapsed, ~0.0s remaining) running mean best_E=-7.203
=== Linear done in 2254.1s ===

=== runn

In [6]:
rows = []
for name, d in results.items():
    if not d["best"]:
        continue
    best_arr, final_arr = np.array(d["best"]), np.array(d["final"])
    rows.append({
        "Schedule": name,
        "Success Probability": np.mean(best_arr == target_E),
        "Mean Best Energy": best_arr.mean(),
        "Std Best Energy": best_arr.std(),
        "Mean Final Energy": final_arr.mean(),
        "Std Final Energy": final_arr.std(),
    })

df = pd.DataFrame(rows)
print(df)

              Schedule  Success Probability  Mean Best Energy  \
0               Linear                0.076            -7.203   
1            Geometric                0.068            -7.187   
2          Logarithmic                0.013            -5.387   
3     Linear Staircase                0.075            -7.157   
4  Geometric Staircase                0.074            -7.182   
5            Quenching                0.007            -5.188   

   Std Best Energy  Mean Final Energy  Std Final Energy  
0         0.886449             -6.907          1.065059  
1         0.842633             -6.708          1.080156  
2         1.080385             -5.386          1.081205  
3         0.884506             -6.859          1.052197  
4         0.877995             -6.707          1.112273  
5         1.094831             -5.187          1.094546  


In [7]:
df.to_csv("sa_schedule_comparison_results.csv", index=False)
np.savez("sa_schedule_raw_results.npz",
         **{f"{name}_best": np.array(d["best"]) for name, d in results.items() if d["best"]},
         **{f"{name}_final": np.array(d["final"]) for name, d in results.items() if d["final"]})
print("saved.")

saved.


In [8]:
target_E = -9
rows = []
for name, d in results.items():
    if not d["best"]:
        continue
    best_arr, final_arr = np.array(d["best"]), np.array(d["final"])
    rows.append({
        "Schedule": name,
        "Success Probability": np.mean(final_arr == target_E),
        "Mean Best Energy": best_arr.mean(),
        "Std Best Energy": best_arr.std(),
        "Mean Final Energy": final_arr.mean(),
        "Std Final Energy": final_arr.std(),
    })
df = pd.DataFrame(rows)
print(df)

              Schedule  Success Probability  Mean Best Energy  \
0               Linear                0.064            -7.203   
1            Geometric                0.053            -7.187   
2          Logarithmic                0.013            -5.387   
3     Linear Staircase                0.065            -7.157   
4  Geometric Staircase                0.054            -7.182   
5            Quenching                0.007            -5.188   

   Std Best Energy  Mean Final Energy  Std Final Energy  
0         0.886449             -6.907          1.065059  
1         0.842633             -6.708          1.080156  
2         1.080385             -5.386          1.081205  
3         0.884506             -6.859          1.052197  
4         0.877995             -6.707          1.112273  
5         1.094831             -5.187          1.094546  


In [11]:
import pickle

with open("all_results_backup.pkl", "wb") as f:
    pickle.dump({
        "df": df,
        "results": results,
    }, f)

print("saved: sa_benchmark_summary.csv, sa_benchmark_raw_trials.csv, all_results_backup.pkl")

saved: sa_benchmark_summary.csv, sa_benchmark_raw_trials.csv, all_results_backup.pkl
